# Previsão de Nível de Burnout em Trabalhadores Remotos

Este notebook realiza a análise exploratória e o treinamento de modelos de machine learning para prever o nível de burnout (Baixo, Médio, Alto) de profissionais, com base em dados demográficos, condições de trabalho e saúde.

**Dataset:** [Remote Work Health Impact Survey 2025](https://www.kaggle.com/datasets/pratyushpuri/remote-work-health-impact-survey-2025) (Kaggle)

**Objetivo:** Treinar e comparar 4 classificadores com otimização de hiperparâmetros via Grid Search e Cross-Validation, e exportar os modelos para uso em uma API Flask.

In [1]:
!pip install mlcroissant

  Using cached mlcroissant-1.0.22-py2.py3-none-any.whl (145 kB)
  Using cached pandas_stubs-2.3.3.260113-py3-none-any.whl (168 kB)


In [5]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import pickle as pkl
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score
import matplotlib.pyplot as plt

In [6]:
# Fetch the Croissant JSON-LD
croissant_dataset = mlc.Dataset('https://www.kaggle.com/datasets/pratyushpuri/remote-work-health-impact-survey-2025/croissant/download')

# Check what record sets are in the dataset
record_sets = croissant_dataset.metadata.record_sets
print(record_sets)

# Fetch the records and put them in a DataFrame
record_set_df = pd.DataFrame(croissant_dataset.records(record_set=record_sets[0].uuid))
record_set_df.head()

SSLError: HTTPSConnectionPool(host='www.kaggle.com', port=443): Max retries exceeded with url: /datasets/pratyushpuri/remote-work-health-impact-survey-2025/croissant/download (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1007)')))

## 1. Coleta e Limpeza dos Dados

O dataset é carregado diretamente do Kaggle utilizando o formato [Croissant (JSON-LD)](https://docs.mlcommons.org/croissant/), um padrão aberto para metadados de datasets de ML que facilita a reprodutibilidade.

As colunas categóricas são armazenadas como bytes pelo Croissant e precisam ser decodificadas para strings. Em seguida, a variável alvo `Burnout_Level` é convertida de texto para valores numéricos ordinais (Low=1, Medium=2, High=3).

In [ ]:
record_set_df.columns

In [ ]:
clean_column_names = []
for c in record_set_df.columns:
  clean_column_names.append(c.split('/')[1])
clean_column_names

In [ ]:
record_set_df.columns = clean_column_names
record_set_df.head()

In [ ]:
for c in [
    "Gender",
    "Region",
    "Industry",
    "Job_Role",
    "Work_Arrangement",
    "Mental_Health_Status",
    "Burnout_Level",
    "Physical_Health_Issues",
    "Salary_Range"
]:
  record_set_df[c] = record_set_df[c].str.decode("utf-8")

In [ ]:
record_set_df.head()

In [ ]:
record_set_df['Burnout_Level'] = record_set_df['Burnout_Level'].apply(
    lambda x:
      int(x.replace('High', '3').replace('Medium', '2').replace('Low', '1'))
    if not isinstance(x, int) else x
    )
record_set_df

In [ ]:
record_set_df.describe()

## 2. Análise Exploratória

Verificamos a distribuição do target `Burnout_Level` para identificar possíveis desbalanceamentos entre as classes. Em seguida, analisamos a média de burnout por categoria de cada feature para entender quais variáveis apresentam maior poder discriminativo.

In [ ]:
burnout_count = record_set_df['Burnout_Level'].value_counts()
plt.bar(burnout_count.index, burnout_count.values)
plt.xlabel('Burnout Level')
plt.ylabel('Count')
plt.show()

In [ ]:
for c in [
    "Region",
    "Industry",
    "Job_Role",
    "Work_Arrangement",
    "Salary_Range",
    "Age",
    "Hours_Per_Week",
    "Work_Life_Balance_Score",
    "Social_Isolation_Score",
]:
  plot_data = record_set_df.groupby(c)['Burnout_Level'].mean().sort_values(ascending=False)
  plt.figure(figsize=(4, 4))
  plt.barh(y=plot_data.index, width=plot_data.values)
  plt.xlim(1,3)
  plt.xlabel('Burnout Level')
  plt.ylabel(c)
  plt.show()

## 3. Pré-processamento

### Codificação de `Physical_Health_Issues`

A coluna `Physical_Health_Issues` contém **múltiplas** condições separadas por "; " (ex: *"Back Pain; Eye Strain"*). Como uma pessoa pode ter mais de um problema de saúde, utilizamos `MultiLabelBinarizer` para transformar cada condição em uma coluna binária independente, ao invés de `get_dummies`, que trataria cada combinação como uma categoria única.

In [ ]:
ph_issues = [r.split('; ') for r in list(record_set_df["Physical_Health_Issues"].unique()) if r]
ph_issues

In [ ]:
enc = MultiLabelBinarizer()
enc.fit(ph_issues)

In [ ]:
enc.classes_

In [ ]:
ph_issues_bin = enc.transform(ph_issues)
ph_issues_bin

In [ ]:
df_ph_issues = pd.DataFrame(
    enc.transform(record_set_df["Physical_Health_Issues"].apply(
        lambda x: x.split('; ') if x else '')
    ), columns=enc.classes_
)
df_ph_issues

### One-Hot Encoding e montagem do dataset final

As demais colunas categóricas (Gender, Region, Industry, Job_Role, Work_Arrangement, Salary_Range) são transformadas via `get_dummies`. Colunas não relevantes para a predição (`Survey_Date`, `Mental_Health_Status`) são descartadas — `Mental_Health_Status` é removida por ter correlação direta com o target, o que causaria data leakage.

In [ ]:
df_drop = record_set_df.drop(columns=["Survey_Date", "Physical_Health_Issues", "Mental_Health_Status"])
df_dummies = pd.get_dummies(
    pd.concat([df_drop, df_ph_issues], axis=1),
    dtype=int
  )
df_dummies.head()

In [ ]:
target = ['Burnout_Level']
Y = df_dummies[target]
X = df_dummies.drop(columns=target)

### Separação treino/teste

A divisão 80/20 é feita **antes** da normalização para garantir que o conjunto de teste permaneça completamente invisível durante o ajuste do scaler e o treinamento dos modelos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

## 4. Treinamento dos Modelos

### Escolha dos classificadores e hiperparâmetros

Quatro algoritmos de naturezas distintas são comparados para avaliar diferentes abordagens de classificação:

| Modelo | Motivo da escolha |
|---|---|
| **KNN** | Baseado em distância; sensível à escala (por isso o MinMaxScaler) |
| **Decision Tree** | Baseado em regras; interpretável e capaz de capturar relações não-lineares |
| **CategoricalNB** | Naive Bayes para features categóricas; funciona bem com dados binarizados |
| **SVC** | Busca margens de separação ótimas; robusto em espaços de alta dimensionalidade |

Para cada modelo, definimos um grid de hiperparâmetros. O `GridSearchCV` testa todas as combinações com **5-fold Cross-Validation**, usando **F1 micro** como métrica — apropriada para classificação multiclasse, pois pondera pelo número de amostras por classe.

In [ ]:
models_params = {
    KNeighborsClassifier(): {
        'n_neighbors': [3, 5, 7, 9],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan'],
    },
    DecisionTreeClassifier(): {
        'max_depth': [None, 5, 10, 20],
        'min_samples_split': [2, 5, 10],
        'criterion': ['gini', 'entropy'],
    },
    CategoricalNB(): {
        'alpha': [0.1, 0.5, 1.0, 2.0],
    },
    SVC(): {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto'],
    },
}

In [ ]:
def pipeline(model, param_grid, X_train, X_test, y_train, y_test):
  grid = GridSearchCV(
      model,
      param_grid,
      cv=5,
      scoring='f1_micro',
      n_jobs=-1,
      refit=True,
  )
  grid.fit(X_train, np.array(y_train).reshape(-1))

  best_model = grid.best_estimator_
  y_pred = best_model.predict(X_test)
  score = f1_score(y_test, y_pred, average='micro')
  accuracy = accuracy_score(y_test, y_pred)

  model_name = best_model.__class__.__name__

  with open(f"{model_name}.pkl", "wb") as f:
    pkl.dump(best_model, f)

  return best_model, grid.best_params_, grid.best_score_, score, accuracy

### Normalização e execução do pipeline

O `MinMaxScaler` normaliza as features para o intervalo [0, 1], essencial para algoritmos sensíveis à escala (KNN e SVC). O scaler é ajustado **apenas no treino** e aplicado ao teste para evitar data leakage.

Os dados de teste e o scaler são exportados como `.pkl` para serem reutilizados nos testes automatizados da API (pytest).

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

with open("scaler.pkl", "wb") as f:
    pkl.dump(scaler, f)

with open("y_test.pkl", "wb") as f:
    pkl.dump(y_test, f)

with open("X_test_scaled.pkl", "wb") as f:
    pkl.dump(X_test_scaled, f)

for model, params in models_params.items():
  print(f"\n{model.__class__.__name__}")
  print(f"  Grid: {params}")
  best_model, best_params, cv_score, test_score, accuracy = pipeline(
      model, params, X_train_scaled, X_test_scaled, y_train, y_test
  )
  print(f"  Melhores parâmetros: {best_params}")
  print(f"  F1 micro (CV): {cv_score:.4f}")
  print(f"  F1 micro (teste): {test_score:.4f}")
  print(f"  Accuracy (teste): {accuracy:.4f}")